# 第三部分：RAII 与资源生命周期

## 实验 1：先复现手动资源管理的问题

本实验先不使用 RAII，而是直接调用 C 文件 API，观察 `std::fopen()` 与 `std::fclose()` 如何依赖程序员手动配对。重点不是文件读写语法，而是找出多个退出路径下资源泄漏的根因。

实验文件统一写入当前目录下的 `outputs/01/`，与第三部分其他实验的输出相互隔离。

### 1. 文件也是一种资源

打开文件会从操作系统获得一个有限资源，使用结束后必须归还：

```text
std::fopen()
     ↓
获得 std::FILE*
     ↓
读写文件
     ↓
std::fclose()
```

只有 `std::fopen()` 成功时才产生需要释放的资源；成功后，每条退出路径都必须恰好调用一次 `std::fclose()`。

In [ ]:
#include <cstdio>
#include <filesystem>
#include <iostream>

### 2. 正常路径看起来没有问题

下面的函数只有打开失败和正常完成两个出口。正常写入后显式关闭文件。

In [ ]:
std::filesystem::create_directories("outputs/01");

const char *manual_file_path =
    "outputs/01/message.txt";

bool write_message_manually(const char *path)
{
    std::FILE *file = std::fopen(path, "w");

    if (file == nullptr)
    {
        std::cerr << "failed to open file" << std::endl;
        return false;
    }

    std::fputs("Hello manual resource management", file);
    std::fclose(file);
    return true;
}

std::cout
    << std::boolalpha
    << write_message_manually(manual_file_path)
    << std::endl;

代码能够工作，但它依赖一个隐藏条件：**以后添加的所有控制流仍然必须经过 `std::fclose()`。** 编译器知道 `file` 是一个指针，却不知道当前函数拥有它，更不知道它必须使用哪个函数释放。

### 3. 提前返回绕过释放

下面故意在写入第一段内容后提前返回。为了让实验可恢复，句柄保存在 `unreleased_file` 中；真实代码如果丢失最后一个指针，通常无法再主动关闭该资源。

In [ ]:
std::FILE *unreleased_file = nullptr;

bool write_with_missing_cleanup(
    const char *path,
    bool something_wrong)
{
    unreleased_file = std::fopen(path, "w");

    if (unreleased_file == nullptr)
    {
        return false;
    }

    std::fputs("first part", unreleased_file);

    if (something_wrong)
    {
        std::cerr << "return before fclose" << std::endl;
        return false; // 泄漏路径
    }

    std::fputs("second part", unreleased_file);
    std::fclose(unreleased_file);
    unreleased_file = nullptr;
    return true;
}

write_with_missing_cleanup(manual_file_path, true);
std::cout
    << "resource still open: "
    << std::boolalpha
    << (unreleased_file != nullptr)
    << std::endl;

执行流在错误分支直接绕过了 `std::fclose()`。操作系统通常会在进程结束时回收文件句柄，但对长期运行的程序来说，这仍然是资源泄漏。先手动恢复本次实验留下的句柄：

In [ ]:
if (unreleased_file != nullptr)
{
    std::fclose(unreleased_file);
    unreleased_file = nullptr;
}

std::cout
    << "resource still open: "
    << std::boolalpha
    << (unreleased_file != nullptr)
    << std::endl;

文件最终被进程退出时的操作系统清理，不代表代码正确。相同问题还会出现在：

- Socket 和文件描述符；
- 数据库连接；
- Mutex 等同步对象；
- GPU buffer；
- C API 返回的 Native handle；
- 动态内存。

这些资源可能更稀缺，或者需要特定释放顺序，泄漏的后果也更难定位。

### 4. C 风格的集中清理

可以通过单一出口减少重复释放：业务分支只设置结果，函数末尾统一清理。C 项目中也常使用 `goto cleanup` 处理多个资源的部分初始化，这不是随意使用 `goto`，而是在语言没有析构函数时人工建立统一生命周期出口。

In [ ]:
bool write_with_single_cleanup(
    const char *path,
    bool something_wrong)
{
    std::FILE *file = std::fopen(path, "w");

    if (file == nullptr)
    {
        return false;
    }

    bool succeeded = false;
    std::fputs("first part", file);

    if (!something_wrong)
    {
        std::fputs("second part", file);
        succeeded = true;
    }

    std::fclose(file); // 唯一清理出口
    return succeeded;
}

std::cout
    << std::boolalpha
    << write_with_single_cleanup(manual_file_path, true)
    << std::endl;

### 5. 为什么集中清理仍然脆弱

单一出口可以工作，但资源越多，清理逻辑越复杂：

```text
获取 A 成功
  ├─ 获取 B 失败 → 只释放 A
  └─ 获取 B 成功
       ├─ 中途失败 → 先释放 B，再释放 A
       └─ 正常完成 → 先释放 B，再释放 A
```

每增加一个资源或退出路径，都要重新审查清理条件和顺序；异常还可能跳过普通控制流中的清理语句。

### 6. 根因：所有权没有被类型表达

`std::FILE*` 只表达一个地址，无法回答谁拥有文件、谁必须关闭、指针能否复制，以及提前返回时由谁清理。当这些规则只存在于注释和开发者记忆中，编译器无法提供帮助。

RAII 的方向是让一个对象拥有 `std::FILE*`：对象构造时获取文件，对象析构时关闭文件，让所有控制流共享 C++ 已有的作用域清理规则。

### 实验结论

手动 `fopen/fclose` 在简单路径上可以工作，但正确性会分散到每个退出分支。随着控制流和资源数量增加，遗漏释放、重复释放和顺序错误都会更容易发生。

下一个实验会把 `std::FILE*` 封装到对象中，让析构函数成为唯一的释放位置。